In [ ]:
%matplotlib inline
import sys, os, json
import torch
import torch.nn as nn
import numpy as np
import torchvision
from torch.utils.data import DataLoader

sys.path.append('..')
from src.data.degredation import get_transforms
from src.models.resnet import resnet18  # CIFAR-native 32x32
from src.training.train import train, test

import matplotlib.pyplot as plt
from custom.figure import mm, color

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams['font.family'] = 'DejaVu Sans'


In [ ]:
dir = "cifar10-native-gradual-lr"
num_net = 10
output_size = 10
lr_target = 0.0001  # normal-phase Adam lr, same as cifarnative_gradual.ipynb

figure_dir = os.path.join("..", "figures", dir)
save_dir = os.path.join("..", "results", dir)
warmup_dir = os.path.join("..", "results", "cifar10-native-0919")  # r warm-up checkpoints
os.makedirs(figure_dir, exist_ok=True)
os.makedirs(save_dir, exist_ok=True)

criterion = nn.CrossEntropyLoss()
batch_size = 128
num_workers = 8

dataset_dir = "../dataset"
train_dataset = torchvision.datasets.CIFAR10(
    root=dataset_dir, train=True, download=True,
    transform=get_transforms("cifar10", blur=0, color=1, test=False))
train_degradation_dataset = torchvision.datasets.CIFAR10(
    root=dataset_dir, train=True, download=True,
    transform=get_transforms("cifar10", blur=7, color=0, test=False))
test_dataset = torchvision.datasets.CIFAR10(
    root=dataset_dir, train=False, download=True,
    transform=get_transforms("cifar10", blur=0, color=1, test=True))
test_degradation_dataset = torchvision.datasets.CIFAR10(
    root=dataset_dir, train=False, download=True,
    transform=get_transforms("cifar10", blur=7, color=0, test=True))

train_degradation_loader = DataLoader(train_degradation_dataset, batch_size=batch_size, shuffle=True,
                                       num_workers=num_workers, persistent_workers=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, persistent_workers=True)
test_degradation_loader = DataLoader(test_degradation_dataset, batch_size=batch_size, shuffle=False,
                                      num_workers=num_workers, persistent_workers=True)

print(f"Train(degraded): {len(train_degradation_dataset)}  "
      f"Test: {len(test_dataset)}  Test(degraded): {len(test_degradation_dataset)}")


## High-initial-lr variant

Same blur-image classification setup as `cifarnative_gradual.ipynb` (load r warm-up
checkpoint, train continuously on degraded CIFAR-10, snapshot at each n in
`n_epochs_list`), except the Adam lr is **`lr_high=0.01` for the first `n_highlr`
epochs, then drops to the normal `lr_target=0.0001`** for the rest of training.
`n_highlr` is the thing this notebook sweeps -- how many initial epochs get the
higher lr before dropping down.

In [ ]:
def warmup_ckpt_path(net_idx, r=10):
    return os.path.join(warmup_dir, f"warmup_native_lr0.1_r{r}_{net_idx}.pth")

def ckpt_path(tag, i):
    return os.path.join(save_dir, f"best_model_{tag}_{i}.pth")

def info_path(tag, i):
    return os.path.join(save_dir, f"training_info_{tag}_{i}.json")


def run_gradual_training_chained_highlr(tag, n_epochs_list, n_highlr, r=10,
                                         lr_high=0.01, lr_low=lr_target, num_net=num_net):
    """Like cifarnative_gradual.ipynb's run_gradual_training_chained, but the
    Adam lr is lr_high for the first n_highlr epochs, then switches to lr_low
    for the remaining epochs (manual two-phase schedule, set directly on the
    optimizer's param_groups -- no StepLR here). Trains continuously up to
    max(n_epochs_list) and snapshots at every n along the way, same as the
    base notebook. Returns {n: [info_net0, info_net1, ...]}."""
    max_epochs = max(n_epochs_list)
    n_set = set(n_epochs_list)
    full_info = []

    for net_idx in range(num_net):
        model = resnet18(num_classes=output_size).to(device)

        warmup_path = warmup_ckpt_path(net_idx, r)
        assert os.path.exists(warmup_path), f"missing warm-up checkpoint: {warmup_path}"
        model.load_state_dict(torch.load(warmup_path, map_location=device))

        optimizer = torch.optim.Adam(model.parameters(), lr=lr_high)
        info = dict(train_loss=[], train_acc=[], blur_test_loss=[], blur_test_acc=[],
                    clean_test_loss=[], clean_test_acc=[], lr=[])

        for epoch in range(max_epochs):
            current_lr = lr_high if epoch < n_highlr else lr_low
            for g in optimizer.param_groups:
                g["lr"] = current_lr

            train_loss, train_acc = train(model, train_degradation_loader, optimizer, criterion, device=device)
            blur_test_loss, blur_test_acc = test(model, test_degradation_loader, criterion, device=device)
            clean_test_loss, clean_test_acc = test(model, test_loader, criterion, device=device)

            info["train_loss"].append(train_loss); info["train_acc"].append(train_acc)
            info["blur_test_loss"].append(blur_test_loss); info["blur_test_acc"].append(blur_test_acc)
            info["clean_test_loss"].append(clean_test_loss); info["clean_test_acc"].append(clean_test_acc)
            info["lr"].append(current_lr)

            n_reached = epoch + 1
            if n_reached in n_set:
                torch.save(model.state_dict(), ckpt_path(f"{tag}_n{n_reached}", net_idx))
                with open(info_path(f"{tag}_n{n_reached}", net_idx), "w") as f:
                    json.dump(info, f)

            print(f"[{tag}] net {net_idx} epoch {n_reached}/{max_epochs} lr={current_lr} "
                  f"train_acc={train_acc:.4f} blur_test_acc={blur_test_acc:.4f} clean_test_acc={clean_test_acc:.4f}")

        full_info.append(info)

    results_by_n = {}
    for n in n_epochs_list:
        results_by_n[n] = [{k: v[:n] for k, v in info.items()} for info in full_info]
    return results_by_n


Quick test: r=5 warm-up, n_highlr=2 (lr=0.01 for 2 epochs then 0.0001), 5 epochs total, num_net=1 -- sanity check + timing.

In [ ]:
import time

t0 = time.time()
test_results = run_gradual_training_chained_highlr("gradual_r5_test", n_epochs_list=[5], n_highlr=2, r=5, num_net=1)
elapsed = time.time() - t0

final_blur_acc = test_results[5][0]["blur_test_acc"][-1]
final_clean_acc = test_results[5][0]["clean_test_acc"][-1]

print(f"elapsed: {elapsed:.1f}s total, {elapsed/5:.1f}s/epoch (train + 2 test evals)")
print(f"final blur test acc:  {final_blur_acc:.4f}")
print(f"final clean test acc: {final_clean_acc:.4f}")


## Main sweep: n_highlr = 1, 2, 3, 5, 10

Edit `n_highlr_list` before running if you want different values -- this is the
axis this notebook is for. Each n_highlr value gets its own full continuous
50-epoch run per net (not shared with the others), so total work scales with
`len(n_highlr_list)`. At ~15s/epoch (num_workers=8) this is roughly
`len(n_highlr_list) * 50 * num_net * 15s` -- e.g. 5 values ~= 10 hours.

In [ ]:
n_epochs_list = [5, 10, 15, 20, 30, 40, 50]
n_highlr_list = [1, 2, 3, 5, 10]

results_by_highlr = {}
for n_highlr in n_highlr_list:
    tag = f"gradual_r10_hl{n_highlr}"
    results_by_highlr[n_highlr] = run_gradual_training_chained_highlr(
        tag, n_epochs_list, n_highlr=n_highlr, r=10)


Summary: final blur test accuracy vs. training length, one line per n_highlr.

In [ ]:
plt.figure(figsize=(70 * mm, 45 * mm))
palette = plt.cm.tab10(np.linspace(0, 1, len(n_highlr_list)))

for n_highlr, col in zip(n_highlr_list, palette):
    sweep_results = results_by_highlr[n_highlr]
    means = [np.mean([info["blur_test_acc"][-1] for info in sweep_results[n]]) for n in n_epochs_list]
    stds = [np.std([info["blur_test_acc"][-1] for info in sweep_results[n]]) for n in n_epochs_list]
    plt.errorbar(n_epochs_list, means, yerr=stds, fmt='o-', color=col, capsize=3,
                 label=f"n_highlr={n_highlr}")

plt.xlabel("training epochs (n, all on blurred images)")
plt.ylabel("final blur test accuracy")
plt.title(f"Blur classification vs training length, by high-lr duration ({num_net} nets)")
plt.legend(fontsize=6)
plt.savefig(os.path.join(figure_dir, "blur_acc_vs_epochs_by_highlr.svg"))
plt.show()

print(f"{'n_highlr':<10}" + "".join(f"n={n:<8}" for n in n_epochs_list))
for n_highlr in n_highlr_list:
    sweep_results = results_by_highlr[n_highlr]
    means = [np.mean([info["blur_test_acc"][-1] for info in sweep_results[n]]) for n in n_epochs_list]
    print(f"{n_highlr:<10}" + "".join(f"{m:<10.4f}" for m in means))
